In [1]:
import pandas as pd
df = pd.read_sql("SELECT timestamp_ms FROM telemetry_samples ORDER BY id", "sqlite:///telemetry.db")
# df.to_csv("telemetry_export.csv", index=False)
print(df.describe())

       timestamp_ms
count  1.207270e+05
mean   8.057936e+08
std    5.806784e+06
min    7.521517e+08
25%    8.061920e+08
50%    8.064179e+08
75%    8.066393e+08
max    8.068620e+08


In [ ]:
import pandas as pd

df = pd.read_sql(
"SELECT timestamp_utc AS timestamp, engine_max_rpm, current_engine_rpm, acceleration_x, acceleration_y, acceleration_z, velocity_x, velocity_y, velocity_z, yaw, pitch, roll, position_x, position_y, position_z, speed, power, torque " \
"FROM telemetry_samples ORDER BY id LIMIT 1400", "sqlite:///telemetry.db")

df['timestamp'] = pd.to_datetime(df["timestamp"], utc=True).astype("int64") // 10**6
df['timestamp'] = df['timestamp'] - min(df['timestamp'])

new_df = df.iloc[::18, :]
print(new_df.shape)

json_file = new_df.to_json('segment_data.json', indent=5)
print(json_file)

(78, 17)
None


In [6]:
import json
import pandas as pd

# Achtung: zwischen engine_max_rpm und current_engine_rpm fehlt ein Komma
df = pd.read_sql(
    """
    SELECT
      timestamp_utc AS timestamp,
      engine_max_rpm,
      current_engine_rpm,
      acceleration_x, acceleration_y, acceleration_z,
      velocity_x, velocity_y, velocity_z,
      yaw, pitch, roll,
      position_x, position_y, position_z,
      speed, power, torque
    FROM telemetry_samples
    ORDER BY id
    LIMIT 1400
    """,
    "sqlite:///telemetry.db",
)

# Timestamp in ms ab Start
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).astype("int64") // 10**6
df["timestamp"] = df["timestamp"] - df["timestamp"].min()

# Downsampling
new_df = df.iloc[::18, :]

# 1) JSON-Liste (ein Array aus Objekten)
records = new_df.to_dict(orient="records")
with open("segment_data.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

# 2) Falls du lieber ein Objekt nach Timestamp willst:
# keyed = new_df.set_index("timestamp").to_dict(orient="index")
# with open("segment_data_by_ts.json", "w", encoding="utf-8") as f:
#     json.dump(keyed, f, ensure_ascii=False, indent=2)

print(f"{len(new_df)} Einträge geschrieben nach segment_data.json")


78 Einträge geschrieben nach segment_data.json


In [ ]:
import pathlib, ollama

text = pathlib.Path("segment_data.json").read_text(encoding="utf-8")

resp = ollama.chat(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "Du bist ein erfahrener Rennfahrer-Coach. Du analysierst Fahrdaten, gibst präzises Feedback zu Linie, Bremspunkten, Gas/Bremse-Dosierung, Lenkung und Setup. Antworte kurz, fokussiert und praxisnah."},
        {"role": "user", "content": f"Analysiere diese Session:\n```json\n{text}\n```"},
    ],
)
print(resp["message"]["content"])


In [8]:
import pathlib, ollama, sys

text = pathlib.Path("segment_data.json").read_text(encoding="utf-8")

messages = [
    {"role": "system", "content": "Du bist ein erfahrener Rennfahrer-Coach. Du analysierst Fahrdaten und gibst präzises Feedback zu Linie, Bremspunkten, Gas/Bremse-Dosierung, Lenkung und Setup. Antworte kurz, fokussiert, praxisnah, maximal 2 Sätze."},
    {"role": "user", "content": f"Analysiere diese Session:\n```json\n{text}\n```"},
]

for chunk in ollama.chat(model="gpt-oss:20b", messages=messages, stream=True):
    sys.stdout.write(chunk["message"]["content"])
    sys.stdout.flush()

The entry and exit lines are clean, but the braking point is a bit early and the throttle application is jerky at high RPM – aim to lift the throttle more gradually as you leave the line to keep traction. Keep the steering input smoother in the high‑speed corners and adjust suspension damping or tire pressures slightly to reduce the slight over‑steer you feel when the car reaches peak power.